In [0]:
%sql
SHOW TABLES IN workspace.bls_productivity;

database,tableName,isTemporary
bls_productivity,bronze_bls_observations,false
bls_productivity,bronze_bls_series,false
bls_productivity,bronze_population,false
bls_productivity,gold_q1_population_statistics,false
bls_productivity,gold_q1_population_statistics_sql,false
bls_productivity,gold_q2_bls_best_year,false
bls_productivity,gold_q2_bls_best_year_sql,false
bls_productivity,gold_q3_prs30006032_population,false
bls_productivity,gold_q3_prs30006032_population_sql,false
bls_productivity,silver_bls_observations,false


In [0]:
%sql
SELECT *
FROM workspace.bls_productivity.gold_q1_population_statistics;

number_of_years,mean_population,sample_standard_deviation,population_standard_deviation,from_year,to_year
6,3.22069808E8,4158441.040908095,3796119.936934378,2013,2018


In [0]:
%sql
SELECT *
FROM workspace.bls_productivity.gold_q1_population_statistics_sql;

number_of_years,mean_population,sample_standard_deviation,population_standard_deviation,from_year,to_year
6,3.22069808E8,4158441.040908095,3796119.936934378,2013,2018


In [0]:
%sql
SELECT
    ABS(p.mean_population - s.mean_population)
        AS mean_difference,
    ABS(
        p.sample_standard_deviation -
        s.sample_standard_deviation
    ) AS stddev_difference
FROM workspace.bls_productivity.gold_q1_population_statistics p
CROSS JOIN workspace.bls_productivity.gold_q1_population_statistics_sql s;

mean_difference,stddev_difference
0.0,0.0


In [0]:
%sql
SELECT
    series_id,
    series_title,
    best_year,
    annual_value_sum,
    quarter_count
FROM workspace.bls_productivity.gold_q2_bls_best_year
ORDER BY series_id;

series_id,series_title,best_year,annual_value_sum,quarter_count
PRS30006011,"Employment for the Manufacturing sector, % Change same quarter 1 year ago",2022,16.4,4
PRS30006012,"Employment for the Manufacturing sector, % Change from previous quarter",2022,13.0,4
PRS30006013,"Employment for the Manufacturing sector, Index (2017=100)",1989,578.369,4
PRS30006014,"Employment for the Manufacturing sector, Level (Millions of jobs)",1989,73.69,4
PRS30006021,"Average weekly hours for the Manufacturing sector, % Change same quarter 1 year ago",2010,14.2,4
PRS30006022,"Average weekly hours for the Manufacturing sector, % Change from previous quarter",2010,8.9,4
PRS30006023,"Average weekly hours for the Manufacturing sector, Index (2017=100)",2014,402.512,4
PRS30006024,"Average weekly hours for the Manufacturing sector, Level (Hours worked per job per week)",2014,155.0,4
PRS30006031,"Hours worked for the Manufacturing sector, % Change same quarter 1 year ago",2022,16.5,4
PRS30006032,"Hours worked for the Manufacturing sector, % Change from previous quarter",1994,14.599999999999998,4


In [0]:
%sql
SELECT
    series_id,
    best_year,
    annual_value_sum
FROM workspace.bls_productivity.gold_q2_bls_best_year

EXCEPT

SELECT
    series_id,
    best_year,
    annual_value_sum
FROM workspace.bls_productivity.gold_q2_bls_best_year_sql;

series_id,best_year,annual_value_sum


In [0]:
%sql
SELECT
    series_id,
    best_year,
    annual_value_sum
FROM workspace.bls_productivity.gold_q2_bls_best_year_sql

EXCEPT

SELECT
    series_id,
    best_year,
    annual_value_sum
FROM workspace.bls_productivity.gold_q2_bls_best_year;

series_id,best_year,annual_value_sum


In [0]:
%sql
SELECT
    series_id,
    year,
    period,
    value,
    population,
    population_available
FROM workspace.bls_productivity.gold_q3_prs30006032_population
ORDER BY year;

series_id,year,period,value,population,population_available
PRS30006032,1988,Q01,2.1,null,false
PRS30006032,1989,Q01,1.8,null,false
PRS30006032,1990,Q01,-4.6,null,false
PRS30006032,1991,Q01,-7.9,null,false
PRS30006032,1992,Q01,-3.1,null,false
PRS30006032,1993,Q01,1.3,null,false
PRS30006032,1994,Q01,1.7,null,false
PRS30006032,1995,Q01,0.0,null,false
PRS30006032,1996,Q01,-4.2,null,false
PRS30006032,1997,Q01,2.8,null,false


In [0]:
%sql
SELECT
    series_id,
    year,
    period,
    value,
    population
FROM workspace.bls_productivity.gold_q3_prs30006032_population

EXCEPT

SELECT
    series_id,
    year,
    period,
    value,
    population
FROM workspace.bls_productivity.gold_q3_prs30006032_population_sql;

series_id,year,period,value,population


In [0]:
%sql
SELECT
    period,
    COUNT(*) AS row_count
FROM workspace.bls_productivity.silver_bls_observations
GROUP BY period
ORDER BY period;

period,row_count
Q01,18162
Q02,18241
Q03,18174
Q04,17958


In [0]:
%sql
-- ============================================================
-- 0. PIPELINE OUTPUT CHECK
-- ============================================================

SHOW TABLES IN workspace.bls_productivity;


-- ============================================================
-- 1. QUESTION 1 - PYSPARK RESULT
-- Mean and standard deviation of US population, 2013-2018
-- ============================================================

SELECT *
FROM workspace.bls_productivity.gold_q1_population_statistics;


-- ============================================================
-- 2. QUESTION 1 - SPARK SQL RESULT
-- ============================================================

SELECT *
FROM workspace.bls_productivity.gold_q1_population_statistics_sql;


-- ============================================================
-- 3. QUESTION 1 - PYSPARK VS SQL VALIDATION
-- Expected differences: 0 or practically 0
-- ============================================================

SELECT
    ABS(
        pyspark_result.mean_population
        - sql_result.mean_population
    ) AS mean_difference,

    ABS(
        pyspark_result.sample_standard_deviation
        - sql_result.sample_standard_deviation
    ) AS sample_stddev_difference,

    ABS(
        pyspark_result.population_standard_deviation
        - sql_result.population_standard_deviation
    ) AS population_stddev_difference

FROM
    workspace.bls_productivity.gold_q1_population_statistics
        AS pyspark_result

CROSS JOIN
    workspace.bls_productivity.gold_q1_population_statistics_sql
        AS sql_result;


-- ============================================================
-- 4. QUESTION 2 - PYSPARK RESULT
-- Best year for every series_id
-- ============================================================

SELECT
    series_id,
    series_title,
    best_year,
    annual_value_sum,
    quarter_count
FROM workspace.bls_productivity.gold_q2_bls_best_year
ORDER BY series_id;


-- ============================================================
-- 5. QUESTION 2 - PYSPARK MINUS SQL
-- Expected: 0 rows
-- ============================================================

SELECT
    series_id,
    best_year,
    annual_value_sum,
    quarter_count
FROM workspace.bls_productivity.gold_q2_bls_best_year

EXCEPT

SELECT
    series_id,
    best_year,
    annual_value_sum,
    quarter_count
FROM workspace.bls_productivity.gold_q2_bls_best_year_sql;


-- ============================================================
-- 6. QUESTION 2 - SQL MINUS PYSPARK
-- Expected: 0 rows
-- This reverse check proves both datasets are equivalent.
-- ============================================================

SELECT
    series_id,
    best_year,
    annual_value_sum,
    quarter_count
FROM workspace.bls_productivity.gold_q2_bls_best_year_sql

EXCEPT

SELECT
    series_id,
    best_year,
    annual_value_sum,
    quarter_count
FROM workspace.bls_productivity.gold_q2_bls_best_year;


-- ============================================================
-- 7. QUESTION 3 - PYSPARK RESULT
-- PRS30006032 / Q01 joined to population
-- ============================================================

SELECT
    series_id,
    year,
    period,
    value,
    population,
    population_available
FROM workspace.bls_productivity.gold_q3_prs30006032_population
ORDER BY year;


-- ============================================================
-- 8. QUESTION 3 - PYSPARK VS SQL
-- Expected: 0 rows
-- ============================================================

SELECT
    series_id,
    year,
    period,
    value,
    population,
    population_available
FROM workspace.bls_productivity.gold_q3_prs30006032_population

EXCEPT

SELECT
    series_id,
    year,
    period,
    value,
    population,
    population_available
FROM workspace.bls_productivity.gold_q3_prs30006032_population_sql;


-- ============================================================
-- 9. SILVER DATA QUALITY CHECK
-- Only quarterly records should survive validation
-- ============================================================

SELECT
    period,
    COUNT(*) AS row_count
FROM workspace.bls_productivity.silver_bls_observations
GROUP BY period
ORDER BY period;


-- ============================================================
-- 10. REJECTED BLS DATA QUALITY SUMMARY
-- ============================================================

SELECT
    rejection_reason,
    COUNT(*) AS rejected_record_count
FROM workspace.bls_productivity.silver_bls_observations_rejected
GROUP BY rejection_reason
ORDER BY rejection_reason;


-- ============================================================
-- 11. FORMAL BLS EXPECTATION OUTPUT
-- ============================================================

SELECT
    COUNT(*) AS expected_bls_row_count
FROM workspace.bls_productivity.silver_bls_observations_expected;


-- ============================================================
-- 12. FORMAL POPULATION EXPECTATION OUTPUT
-- ============================================================

SELECT
    COUNT(*) AS expected_population_row_count
FROM workspace.bls_productivity.silver_us_population_expected;

expected_population_row_count
11
